In [26]:
import pandas as pd
import tensorflow as tf
import sklearn
from tensorflow.keras.layers import Dense, Dropout, Activation, Input
from tensorflow.keras.models import Model
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error #imports

In [27]:
base = pd.read_csv('games.csv') #lendo csv
base

,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8,322.0,Nintendo,E
1,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NaN,NaN,NaN,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E
3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8,192.0,Nintendo,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16714,Samurai Warriors: Sanada Maru,PS3,2016.0,Action,Tecmo Koei,0.00,0.00,0.01,0.00,0.01,NaN,NaN,NaN,NaN,NaN,NaN
16715,LMA Manager 2007,X360,2006.0,Sports,Codemasters,0.00,0.01,0.00,0.00,0.01,NaN,NaN,NaN,NaN,NaN,NaN
16716,Haitaka no Psychedelica,PSV,2016.0,Adventure,Idea Factory,0.00,0.00,0.01,0.00,0.01,NaN,NaN,NaN,NaN,NaN,NaN
16717,Spirits & Spells,GBA,2003.0,Platform,Wanadoo,0.01,0.00,0.00,0.00,0.01,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
base = base.drop('Other_Sales', axis = 1) #remove uma coluna

In [29]:
base = base.drop('Global_Sales', axis = 1)
base = base.drop('Developer', axis = 1) #remove mais 2 colunas

In [30]:
base.shape #ve o shape

(16719, 13)

In [31]:
base.isnull().sum() #vendo quais linhas masi tem null

Name                  2
Platform              0
Year_of_Release     269
Genre                 2
Publisher            54
NA_Sales              0
EU_Sales              0
JP_Sales              0
Critic_Score       8582
Critic_Count       8582
User_Score         6704
User_Count         9129
Rating             6769
dtype: int64

In [32]:
base = base.dropna(axis = 0) #apagando as linhas que tem nulo

In [33]:
base.shape

(6825, 13)

In [34]:
base.isnull().sum() #vendo quais linhas masi tem null

Name               0
Platform           0
Year_of_Release    0
Genre              0
Publisher          0
NA_Sales           0
EU_Sales           0
JP_Sales           0
Critic_Score       0
Critic_Count       0
User_Score         0
User_Count         0
Rating             0
dtype: int64

In [35]:
base['Name'].value_counts() #vendo os nomes mais frequentes

Name
Madden NFL 07                                   8
Need for Speed: Most Wanted                     8
LEGO Star Wars II: The Original Trilogy         8
Terraria                                        7
Madden NFL 08                                   7
                                               ..
Brain Age: Train Your Brain in Minutes a Day    1
Wii Fit                                         1
Wii Sports Resort                               1
Mario Kart Wii                                  1
Wii Sports                                      1
Name: count, Length: 4377, dtype: int64

In [36]:
base = base.drop('Name', axis = 1) #apagando os nomes

In [37]:
base.shape #formato da base

(6825, 12)

In [38]:
base.columns #vendo as colunas

Index(['Platform', 'Year_of_Release', 'Genre', 'Publisher', 'NA_Sales',
       'EU_Sales', 'JP_Sales', 'Critic_Score', 'Critic_Count', 'User_Score',
       'User_Count', 'Rating'],
      dtype='object')

In [39]:
X = base.iloc[:, [0, 1, 2, 3, 7, 8, 9, 10, 11]].values #pegando algumas colunas

In [40]:
y_na = base.iloc[:, 4].values
y_eu = base.iloc[:, 5].values
y_jp = base.iloc[:, 6].values #EUA EUROPA E JAPAO

In [41]:
y_na, y_eu, y_jp #vendo as variaveis

(array([4.136e+01, 1.568e+01, 1.561e+01, ..., 0.000e+00, 1.000e-02,
        0.000e+00], shape=(6825,)),
 array([2.896e+01, 1.276e+01, 1.093e+01, ..., 1.000e-02, 0.000e+00,
        1.000e-02], shape=(6825,)),
 array([3.77, 3.79, 3.28, ..., 0.  , 0.  , 0.  ], shape=(6825,)))

In [42]:
base['Platform'].value_counts() #vendo as plataformas

Platform
PS2     1140
X360     858
PS3      769
PC       651
XB       565
Wii      479
DS       464
PSP      390
GC       348
PS4      239
GBA      237
XOne     159
3DS      155
PS       150
PSV      118
WiiU      89
DC        14
Name: count, dtype: int64

In [43]:
onechotencoder = ColumnTransformer(transformers=[("OneHot", OneHotEncoder(), [0, 2, 3, 8])], remainder='passthrough') #tirando os dados categoricos e jogando pra numero

In [44]:
X = onechotencoder.fit_transform(X).toarray() #jogando pra X

In [45]:
X.shape #formato

(6825, 303)

In [46]:
X[0] #primeiro registro

array([0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 1.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 

In [51]:
camada_entrada = Input(shape = (303,))
camada_oculta1 = Dense(units = 153, activation = 'relu')(camada_entrada)
camada_oculta2 = Dense(units = 153, activation = 'relu')(camada_oculta1)
camada_saida1 = Dense(units = 1, activation = 'linear')(camada_oculta2)
camada_saida2 = Dense(units = 1, activation = 'linear')(camada_oculta2)
camada_saida3 = Dense(units = 1, activation = 'linear')(camada_oculta2) #fazendo as camadas

In [52]:
regressor = Model(inputs = camada_entrada, outputs = [camada_saida1, camada_saida2, camada_saida3]) #cria o modelo

In [53]:
regressor.compile(optimizer = 'adam', loss = 'mse') #compila

In [54]:
regressor.fit(X, [y_na, y_eu, y_jp], epochs = 500, batch_size = 100) #treinando

Epoch 1/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - dense_10_loss: 65.1360 - dense_11_loss: 85.0194 - dense_9_loss: 172.0341 - loss: 325.5493  
Epoch 2/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - dense_10_loss: 5.6647 - dense_11_loss: 10.4431 - dense_9_loss: 3.6819 - loss: 19.9649
Epoch 3/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - dense_10_loss: 6.1204 - dense_11_loss: 11.2349 - dense_9_loss: 10.0217 - loss: 27.5834
Epoch 4/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - dense_10_loss: 4.4957 - dense_11_loss: 1.2502 - dense_9_loss: 2.6176 - loss: 8.4028 
Epoch 5/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - dense_10_loss: 20.8415 - dense_11_loss: 2.3750 - dense_9_loss: 9.1225 - loss: 32.6446
Epoch 6/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - dense_10_loss: 26.1207 - dense_11_loss: 3.3598 - dense_9_loss: 11.0019 - loss: 40.9183
Epoch 7/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - dense_10_loss: 19.0432 - dense_11_loss: 44.6427 - dense_9_loss: 62.4431 - loss: 127.2902
Epoch 8/500
69/69

In [56]:
previsao_na, previsao_eu, previsao_jp = regressor.predict(X) #previsao

214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 567us/step


In [58]:
previsao_na, previsao_na.mean() #vendo as previsao pros EUA

(array([[ 9.398455  ],
        [11.906643  ],
        [ 6.8625307 ],
        ...,
        [ 0.19955532],
        [ 0.13719617],
        [ 0.09154147]], shape=(6825, 1), dtype=float32),
 np.float32(0.77845126))

In [59]:
y_na, y_na.mean() #vendo os valores reais

(array([4.136e+01, 1.568e+01, 1.561e+01, ..., 0.000e+00, 1.000e-02,
        0.000e+00], shape=(6825,)),
 np.float64(0.3944835164835165))

In [62]:
mean_absolute_error(y_na, previsao_na) #vendo a media do erro abosluto

0.5161662807281201

In [63]:
previsao_eu, previsao_eu.mean() #previsao europa

(array([[6.794726  ],
        [8.641313  ],
        [4.6481113 ],
        ...,
        [0.23893823],
        [0.08127278],
        [0.04301485]], shape=(6825, 1), dtype=float32),
 np.float32(0.47745767))

In [67]:
y_eu, y_eu.mean() #valo real europa

(array([2.896e+01, 1.276e+01, 1.093e+01, ..., 1.000e-02, 0.000e+00,
        1.000e-02], shape=(6825,)),
 np.float64(0.23608937728937732))

In [68]:
mean_absolute_error(y_eu, previsao_eu) #vendo a media do erro abosluto

0.3499903739897759

In [65]:
previsao_jp, previsao_jp.mean() #media da previsao japao

(array([[3.1995451 ],
        [3.913171  ],
        [2.0404584 ],
        ...,
        [0.07936634],
        [0.04691871],
        [0.00900094]], shape=(6825, 1), dtype=float32),
 np.float32(0.1758176))

In [66]:
y_jp, y_jp.mean() #valor real japao

(array([3.77, 3.79, 3.28, ..., 0.  , 0.  , 0.  ], shape=(6825,)),
 np.float64(0.06415824175824175))

In [69]:
mean_absolute_error(y_jp, previsao_jp) #vendo a media do erro abosluto

0.16751668154581997